# 🧠 3D VisReg JEPA (8³ Token Grid) Pre-training on NVIDIA A100
### BraTS 2024 Adult Glioma Volumetric Pre-training Suite

This notebook executes self-supervised 3D VisReg JEPA pre-training on **Google Colab with an A100 GPU**.

**Architectural & Algorithmic Highlights**:
1. **8³ Token Grid (512 tokens)**: 128x128x128 MRI volume partitioned into non-overlapping 16x16x16 patch cuboids.
2. **Option B (3D Anisotropic Target Masking Jitter)**: Multi-aspect ratio target slabs (axial, coronal, sagittal, and meso blocks) reflecting realistic glioma morphology.
3. **Upgrade C (Cosine Regularizer Annealing Schedule)**: Preserves variance/covariance regularization scale at 1.0 for 15 epochs, then cosine-anneals down to 0.10 to prevent late-stage over-constraining.
4. **A100 Acceleration**: High-throughput training (`batch_size=16`, AMP FP16/BF16, CUDA 12) taking ~1–1.5 hours for 50 epochs on 1,144 volumes.
5. **Private Kaggle Dataset & Drive Persistence**: Directly downloads your private Kaggle dataset to Colab local NVMe (`/content/data/`) and streams checkpoints to Google Drive (`/content/drive/MyDrive/checkpoints/`).


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time
import torch

print(f"⏱ Session Start: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

!nvidia-smi

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device Name:      {gpu_name}")
    print(f"Total VRAM:       {vram_gb:.2f} GB")
    if "A100" not in gpu_name:
        print("⚠️ WARNING: You are not running on an A100 GPU. Go to Runtime -> Change runtime type -> A100 GPU for optimal throughput.")
    else:
        print("✓ A100 GPU confirmed. Ready for high-throughput 3D volumetric training.")
else:
    raise RuntimeError("No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU.")


## 2. Mount Google Drive (Checkpoint Safety)
Mounting Google Drive ensures that checkpoints, training logs, and exported weights are saved permanently and are not lost if the Colab tab is closed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")


## 3. Private Kaggle Dataset Download
Download the preprocessed BraTS 3D dataset directly into Colab's fast local NVMe disk (`/content/data/`).

> **Authentication**: Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the **Colab Secrets** tab (🔑 on the left sidebar), or upload your `kaggle.json` when prompted.


In [ ]:
import os
from pathlib import Path

# 1. Set your private Kaggle dataset handle (e.g., 'username/dataset-slug')
KAGGLE_DATASET = "hanriman/brats-3d-full"  # <-- UPDATE IF NEEDED

# 2. Authenticate Kaggle API
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('✓ Authenticated Kaggle API via Colab Secrets.')
except Exception:
    pass

if not os.environ.get('KAGGLE_KEY'):
    from google.colab import files
    print('Colab Secret not found. Please upload your kaggle.json:')
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    print('✓ Authenticated Kaggle API via kaggle.json.')

# 3. Fast download & unzip into local NVMe storage
!mkdir -p /content/data
!kaggle datasets download -d {KAGGLE_DATASET} -p /content/data --unzip

# Locate data directory
cands = [Path('/content/data/brats_gli_3d_v2'), Path('/content/data/brats_gli_3d_full'), Path('/content/data')]
data_dir = next((p for p in cands if p.is_dir() and (p / 'metadata.csv').exists()), Path('/content/data'))
os.environ['BRATS3D_DATA_DIR'] = str(data_dir)
print(f"✓ Dataset ready at: {data_dir} ({len(list(data_dir.glob('*.npz')))} volumes found)")


## 4. Clone Repository & Install Dependencies


In [ ]:
import os
from pathlib import Path

%cd /content
if not Path('/content/tumor_segmentation_3d').exists():
    !git clone https://github.com/hanriman/tumor_segmentation_3d.git
%cd /content/tumor_segmentation_3d
!git fetch origin && git checkout main && git pull origin main

!pip install -q -e .
!pip install -q monai nibabel tabulate matplotlib
print("✓ Dependencies installed and environment configured on branch 'main'.")


## 5. Pre-Flight Verification Smoke Test (2 Minutes)
Verify the entire pre-training pipeline (CUDA, AMP FP16, Anisotropic Masking Jitter, Cosine Regularizer Annealing) on 2 mini-batches before starting the full 50-epoch run.


In [ ]:
!python scripts/train_jepa_3d.py \
    --data_dir {os.environ.get('BRATS3D_DATA_DIR', '/content/data')} \
    --model_type visreg_jepa \
    --smoke_test \
    --reg_warmup_epochs 2 \
    --reg_min_scale 0.10


## 6. Full 3D VisReg JEPA Pre-training on A100 (50 Epochs, BS=16)
Trains the 3D Vision Transformer backbone (8³ = 512 token grid) with:
- **Batch size 16** (high A100 memory utilization)
- **Option B Anisotropic Jitter Masking** (multi-aspect ratio target slabs)
- **Upgrade C Cosine Annealing Schedule** (anneals regularizer scale from 1.0 to 0.10 after epoch 15)
- **Google Drive Checkpointing** (`/content/drive/MyDrive/checkpoints/visreg_jepa_a100_50ep`)


In [ ]:
from pathlib import Path

GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/checkpoints/visreg_jepa_a100_50ep"
Path(GDRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

!python scripts/train_jepa_3d.py \
    --data_dir {os.environ.get('BRATS3D_DATA_DIR', '/content/data')} \
    --model_type visreg_jepa \
    --exp_config configs/experiment/colab_a100_pretrain.yaml \
    --output_dir {GDRIVE_OUTPUT_DIR}


## 7. Post-Training Diagnostics & Artifact Packaging


In [ ]:
from pathlib import Path
from brats_jepa_3d.utils import export_artifacts

# 1. Run Brain-Aware Masking Diagnostic on 200 real test volumes
!python scripts/diagnose_visreg_3d.py --num_volumes 200

# 2. Package checkpoint and logs directly to Google Drive
zip_dest = Path("/content/drive/MyDrive/checkpoints/visreg_pretrain_outputs_a100.zip")
export_res = export_artifacts(
    export_name="visreg_pretrain_outputs_a100",
    export_dir=Path("/content/export_visreg_pretrain"),
    zip_path=zip_dest,
    model_prefix="visreg",
    verbose=True,
)
print(f"✓ Checkpoint and diagnostic package saved to Google Drive: {zip_dest}")
